# Experiment A.04 — full serial run
Launch ID first. Launch OOD only after ID is complete and GPU 1 is idle.

In [ ]:
import os,subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; N=Path.home()/"stage1-native"; OUT=Path.home()/"experiment_a"; MAN=OUT/"experiment_a_manifest.csv"; SCENE="id"; PY=Path.home()/"venv-stage1-id/bin/python"; log=OUT/"experiment_a_id.log"; pidfile=OUT/"experiment_a_id.pid"
env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":"1","MUJOCO_EGL_DEVICE_ID":"1","MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_experiment_a","--config",str(R/"async_vla_benchmark/configs/experiment_a.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene",SCENE,"--resume","--verbose"]
fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)+"\n"); print("launched ID",proc.pid,log)

In [ ]:
import csv
def alive(pid):
 state=subprocess.run(["ps","-p",str(pid),"-o","stat="],capture_output=True,text=True).stdout.strip(); return bool(state) and not state.startswith("Z")
rows=list(csv.DictReader(open(OUT/"experiment_a_episode_results.csv"))) if (OUT/"experiment_a_episode_results.csv").exists() else []; ids={r["run_id"] for r in rows}; print(f"episodes {len(ids)}/64; remaining {64-len(ids)}")
for scene in ("id","ood"):
 p=OUT/f"experiment_a_{scene}.pid"; pid=int(p.read_text()) if p.exists() else -1; print(scene,"alive=",alive(pid),"pid=",pid); log=OUT/f"experiment_a_{scene}.log"; print("\n".join(log.read_text(errors="replace").splitlines()[-8:]) if log.exists() else "not started")

In [ ]:
rows=list(csv.DictReader(open(OUT/"experiment_a_episode_results.csv"))) if (OUT/"experiment_a_episode_results.csv").exists() else []; id_rows=[r for r in rows if r["scene_condition"]=="id" and r.get("status","").startswith("ok")]; assert len({r["run_id"] for r in id_rows})==16,"STOP: ID must be 16/16 valid before OOD"
used,util=[int(x.strip()) for x in subprocess.run(["nvidia-smi","-i","1","--query-gpu=memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.split(',')]; assert used<500 and util<5,"STOP: GPU is not idle"
SCENE="ood"; PY=Path.home()/"venv-stage1-ood/bin/python"; log=OUT/"experiment_a_ood.log"; pidfile=OUT/"experiment_a_ood.pid"; env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":"1","MUJOCO_EGL_DEVICE_ID":"1","MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(P),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+env.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+env.get("LD_LIBRARY_PATH","")})
cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_experiment_a","--config",str(R/"async_vla_benchmark/configs/experiment_a.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene",SCENE,"--resume","--verbose"]
fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)+"\n"); print("launched OOD",proc.pid,log)

In [ ]:
rows=list(csv.DictReader(open(OUT/"experiment_a_episode_results.csv"))) if (OUT/"experiment_a_episode_results.csv").exists() else []; ids={r["run_id"] for r in rows}; print(f"episodes {len(ids)}/64; remaining {64-len(ids)}")
for scene in ("id","ood"):
 p=OUT/f"experiment_a_{scene}.pid"; pid=int(p.read_text()) if p.exists() else -1; print(scene,"alive=",alive(pid),"pid=",pid); log=OUT/f"experiment_a_{scene}.log"; print("\n".join(log.read_text(errors="replace").splitlines()[-8:]) if log.exists() else "not started")
print("Checkpoint ~/experiment_a off-machine regularly. Rerun the matching launch cell after interruption; --resume skips valid episodes.")